# MSCNet Baseline Training (Phase 0)

**AGMS-Net MTP Project** — Reproducing MSCNet (IEEE TSC 2025) on Alibaba Cluster Trace 2018

This notebook:
1. Clones the project repo
2. Installs dependencies
3. Uploads the preprocessed `alibaba.csv` dataset
4. Trains MSCNet for all 4 prediction horizons (H=24, 48, 72, 96)
5. Reports MSE/MAE/RMSE and compares against paper values

**Runtime**: Make sure you select **GPU** runtime (Runtime > Change runtime type > T4 GPU)

## 1. Setup: Check GPU, Clone Repo, Install Dependencies

In [ ]:
# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone the project repo
import os
if not os.path.exists('/content/Workload_Prediction_MTP'):
    !git clone https://github.com/patelkanak23/Workload_Prediction_MTP.git /content/Workload_Prediction_MTP
else:
    !cd /content/Workload_Prediction_MTP && git pull

# Change to the MSCNet baseline directory (where run.py lives)
%cd /content/Workload_Prediction_MTP/MSCNet_baseline
print('\nWorking directory:', os.getcwd())
print('run.py exists:', os.path.exists('run.py'))
print('exp_basic.py exists:', os.path.exists('exp/exp_basic.py'))

In [ ]:
# Install missing dependencies (Colab has torch, numpy, pandas, sklearn pre-installed)
!pip install -q einops ptflops torchinfo reformer-pytorch PyWavelets sktime patool matplotlib tqdm

## 2. Upload Dataset

Upload `alibaba.csv` (1.8 MB) from your local machine.

In [ ]:
import os
import pandas as pd

# Create dataset directory
os.makedirs('dataset', exist_ok=True)

if os.path.exists('dataset/alibaba.csv'):
    print('Dataset already exists!')
else:
    from google.colab import files
    print('Please upload alibaba.csv (1.8 MB):')
    uploaded = files.upload()
    for filename in uploaded:
        os.rename(filename, 'dataset/alibaba.csv')
        print(f'Saved to dataset/alibaba.csv')

# Verify
df = pd.read_csv('dataset/alibaba.csv')
print(f'\nDataset: {df.shape}')
print(f'Columns: {list(df.columns[:4])} ... {list(df.columns[-3:])}')
print(f'Time: {df["date"].iloc[0]} to {df["date"].iloc[-1]}')
print(f'\nReady to train!')

## 3. Train MSCNet — All 4 Horizons

Using exact hyperparameters from `scripts/alibaba.sh`:
- `seq_len=96, label_len=96, features=M, enc_in=100`
- `e_layers=3, n_heads=4, d_model=128`
- `train_epochs=100, patience=3`

In [ ]:
# Setup directories
import os
os.makedirs('logs/long_term_forecast_workload', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
print('Working directory:', os.getcwd())
print('Dataset exists:', os.path.exists('dataset/alibaba.csv'))

In [ ]:
# ========== Train H=24 ==========
!python -u run.py \
    --task_name long_term_forecast \
    --is_training 1 \
    --root_path ./dataset \
    --data_path alibaba.csv \
    --model_id alibaba_96_24 \
    --model MSCNet \
    --data custom \
    --features M \
    --freq t \
    --seq_len 96 \
    --label_len 96 \
    --pred_len 24 \
    --e_layers 3 \
    --d_layers 1 \
    --factor 3 \
    --enc_in 100 \
    --dec_in 100 \
    --c_out 100 \
    --des Exp \
    --itr 1 \
    --n_heads 4 \
    --d_model 128 \
    --train_epochs 100 \
    --patience 3

In [ ]:
# ========== Train H=48 ==========
!python -u run.py \
    --task_name long_term_forecast \
    --is_training 1 \
    --root_path ./dataset \
    --data_path alibaba.csv \
    --model_id alibaba_96_48 \
    --model MSCNet \
    --data custom \
    --features M \
    --freq t \
    --seq_len 96 \
    --label_len 96 \
    --pred_len 48 \
    --e_layers 3 \
    --d_layers 1 \
    --factor 3 \
    --enc_in 100 \
    --dec_in 100 \
    --c_out 100 \
    --des Exp \
    --itr 1 \
    --n_heads 4 \
    --d_model 128 \
    --train_epochs 100 \
    --patience 3

In [ ]:
# ========== Train H=72 ==========
!python -u run.py \
    --task_name long_term_forecast \
    --is_training 1 \
    --root_path ./dataset \
    --data_path alibaba.csv \
    --model_id alibaba_96_72 \
    --model MSCNet \
    --data custom \
    --features M \
    --freq t \
    --seq_len 96 \
    --label_len 96 \
    --pred_len 72 \
    --e_layers 3 \
    --d_layers 1 \
    --factor 3 \
    --enc_in 100 \
    --dec_in 100 \
    --c_out 100 \
    --des Exp \
    --itr 1 \
    --n_heads 4 \
    --d_model 128 \
    --train_epochs 100 \
    --patience 3

In [ ]:
# ========== Train H=96 ==========
!python -u run.py \
    --task_name long_term_forecast \
    --is_training 1 \
    --root_path ./dataset \
    --data_path alibaba.csv \
    --model_id alibaba_96_96 \
    --model MSCNet \
    --data custom \
    --features M \
    --freq t \
    --seq_len 96 \
    --label_len 96 \
    --pred_len 96 \
    --e_layers 3 \
    --d_layers 1 \
    --factor 3 \
    --enc_in 100 \
    --dec_in 100 \
    --c_out 100 \
    --des Exp \
    --itr 1 \
    --n_heads 4 \
    --d_model 128 \
    --train_epochs 100 \
    --patience 3

## 4. Results Summary

In [ ]:
import numpy as np
import os
import glob

# Paper reference values (Alibaba, L=96)
paper_values = {
    24: {'MSE': 0.0030, 'MAE': 0.0292, 'RMSE': 0.0544},
    48: {'MSE': 0.0041, 'MAE': 0.0356, 'RMSE': 0.0637},
    72: {'MSE': 0.0050, 'MAE': 0.0399, 'RMSE': 0.0707},
    96: {'MSE': 0.0058, 'MAE': 0.0433, 'RMSE': 0.0761},
}

print('=' * 85)
print('MSCNet Reproduction Results vs. Paper (Alibaba, L=96)')
print('=' * 85)
print(f'{"H":>4} | {"Our MSE":>10} | {"Paper MSE":>10} | {"Delta%":>7} | {"Our MAE":>10} | {"Paper MAE":>10} | {"Delta%":>7}')
print('-' * 85)

# Find result directories
results_dirs = sorted(glob.glob('results/*'))

found_any = False
for d in results_dirs:
    metrics_path = os.path.join(d, 'metrics.npy')
    if os.path.exists(metrics_path):
        metrics = np.load(metrics_path)
        mae, mse, rmse = metrics[0], metrics[1], metrics[2]
        
        for h in [24, 48, 72, 96]:
            if f'pl{h}_' in d:
                paper = paper_values[h]
                mse_delta = (mse - paper['MSE']) / paper['MSE'] * 100
                mae_delta = (mae - paper['MAE']) / paper['MAE'] * 100
                print(f'{h:>4} | {mse:>10.6f} | {paper["MSE"]:>10.6f} | {mse_delta:>+6.1f}% | {mae:>10.6f} | {paper["MAE"]:>10.6f} | {mae_delta:>+6.1f}%')
                found_any = True
                break

print('=' * 85)

if not found_any:
    print('\nNo results found yet. Result directories:', results_dirs)
    alt = glob.glob('**/metrics.npy', recursive=True)
    print('All metrics.npy files:', alt)
else:
    print('\nDeviations within +/-15% are normal for reproductions.')

In [ ]:
# Print raw results log
if os.path.exists('result_long_term_forecast.txt'):
    print('--- Raw Results Log ---')
    with open('result_long_term_forecast.txt', 'r') as f:
        print(f.read())
else:
    print('No result log found yet.')

## 5. Save Results to Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

save_dir = '/content/drive/MyDrive/AGMS_Net_Results/Phase0'
os.makedirs(save_dir, exist_ok=True)

for folder in ['results', 'checkpoints']:
    if os.path.exists(folder):
        shutil.copytree(folder, f'{save_dir}/{folder}', dirs_exist_ok=True)
        print(f'Saved {folder}/ to Drive')

if os.path.exists('result_long_term_forecast.txt'):
    shutil.copy('result_long_term_forecast.txt', f'{save_dir}/')
    print('Saved result log to Drive')

print(f'\nAll saved to: {save_dir}')